In [3]:
# One-cell dataset audit + sample visualization + class analysis + PyTorch Dataset
# Works for: source/dataset/foodseg103/{train,test}/{img,ann,mask}

import os, json, random, math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset

# =========================
# 0) CONFIG
# =========================
ROOT = Path("/dataset/foodseg103")
if not ROOT.exists():
    ROOT = Path("dataset/foodseg103")
SPLITS = ["train", "test"]
SEED = 42
N_VIS = 3                  # số sample visualize mỗi split
MAX_MASKS_FOR_FULL_SCAN = None   # None = scan full; hoặc đặt số như 200 để scan nhanh
MAKE_VAL_FROM_TRAIN = True
VAL_RATIO = 0.1

random.seed(SEED)
np.random.seed(SEED)

# =========================
# 1) PATHS + BASIC CHECK
# =========================
assert ROOT.exists(), f"Không thấy thư mục: {ROOT}"

meta_path = ROOT / "meta.json"
mapping_path = ROOT / "class_mapping.json"

In [4]:


print("=" * 100)
print("DATASET ROOT:", ROOT.resolve())
print("=" * 100)

for split in SPLITS:
    img_dir  = ROOT / split / "img"
    ann_dir  = ROOT / split / "ann"
    mask_dir = ROOT / split / "mask"
    print(f"[{split}]")
    print("  img :", img_dir,  "exists=", img_dir.exists())
    print("  ann :", ann_dir,  "exists=", ann_dir.exists())
    print("  mask:", mask_dir, "exists=", mask_dir.exists())
    if img_dir.exists():
        print("  #img :", len(list(img_dir.glob("*"))))
    if ann_dir.exists():
        print("  #ann :", len(list(ann_dir.glob("*.json"))))
    if mask_dir.exists():
        print("  #mask:", len(list(mask_dir.glob("*.png"))))
    print()

# =========================
# 2) LOAD META + CLASS MAPPING
# =========================
meta = None
if meta_path.exists():
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

mapping = None
if mapping_path.exists():
    with open(mapping_path, "r", encoding="utf-8") as f:
        mapping = json.load(f)

print("=" * 100)
print("META / CLASS MAPPING")
print("=" * 100)

if meta is not None:
    meta_classes = meta.get("classes", [])
    meta_tags = meta.get("tags", [])
    print(f"meta.json found")
    print(f"  num meta classes : {len(meta_classes)}")
    print(f"  num meta tags    : {len(meta_tags)}")
    if len(meta_classes) > 0:
        print("  first 10 meta class titles:", [c.get("title") for c in meta_classes[:10]])
else:
    print("meta.json not found")

if mapping is not None:
    class_to_id = mapping["class_to_id"]
    id_to_class = {int(k): v for k, v in mapping["id_to_class"].items()}
    background_id = int(mapping["background_id"])
    num_ingredient_classes = int(mapping["num_ingredient_classes"])
    num_classes = num_ingredient_classes + 1
    print(f"class_mapping.json found")
    print(f"  num ingredient classes : {num_ingredient_classes}")
    print(f"  background_id          : {background_id}")
    print(f"  num_classes(total)     : {num_classes}")
    print("  first 10 classes:", list(class_to_id.items())[:10])
else:
    # fallback: try from meta.json
    if meta is None:
        raise RuntimeError("Không có cả meta.json lẫn class_mapping.json")
    class_titles = sorted([c["title"] for c in meta["classes"]])
    class_to_id = {name: i for i, name in enumerate(class_titles)}
    id_to_class = {i: name for name, i in class_to_id.items()}
    background_id = len(class_titles)
    num_ingredient_classes = len(class_titles)
    num_classes = num_ingredient_classes + 1
    print("Không có class_mapping.json -> fallback từ meta.json")
    print(f"  num ingredient classes : {num_ingredient_classes}")
    print(f"  background_id          : {background_id}")
    print(f"  num_classes(total)     : {num_classes}")

# =========================
# 3) FILE MATCHING CHECK
# =========================
def stem_without_double_ext(p: Path):
    # 00000001.jpg.json -> 00000001
    name = p.name
    if name.endswith(".jpg.json"):
        return name[:-9]
    if name.endswith(".png.json"):
        return name[:-9]
    return p.stem

print("\n" + "=" * 100)
print("FILE MATCHING CHECK")
print("=" * 100)

split_files = {}
for split in SPLITS:
    img_dir  = ROOT / split / "img"
    ann_dir  = ROOT / split / "ann"
    mask_dir = ROOT / split / "mask"

    img_stems  = {p.stem for p in img_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]}
    ann_stems  = {stem_without_double_ext(p) for p in ann_dir.glob("*.json")}
    mask_stems = {p.stem for p in mask_dir.glob("*.png")} if mask_dir.exists() else set()

    common_img_ann = sorted(img_stems & ann_stems)
    common_all = sorted(img_stems & ann_stems & mask_stems) if mask_dir.exists() else common_img_ann

    print(f"[{split}]")
    print(f"  img stems : {len(img_stems)}")
    print(f"  ann stems : {len(ann_stems)}")
    print(f"  mask stems: {len(mask_stems)}")
    print(f"  common img-ann     : {len(common_img_ann)}")
    print(f"  common img-ann-mask: {len(common_all)}")
    if len(img_stems - ann_stems) > 0:
        print("  sample img without ann :", list(sorted(img_stems - ann_stems))[:5])
    if len(ann_stems - img_stems) > 0:
        print("  sample ann without img :", list(sorted(ann_stems - img_stems))[:5])
    if mask_dir.exists() and len(img_stems - mask_stems) > 0:
        print("  sample img without mask:", list(sorted(img_stems - mask_stems))[:5])
    split_files[split] = common_all
    print()

# =========================
# 4) SIZE + LABEL DISTRIBUTION SCAN
# =========================
print("=" * 100)
print("SCANNING IMAGE/MASK STATS")
print("=" * 100)

img_size_stats = {}
mask_value_stats = {}
pixel_counts_per_class = {}
image_counts_per_class = {}
sample_records = defaultdict(list)

for split in SPLITS:
    files = split_files[split]
    if MAX_MASKS_FOR_FULL_SCAN is not None:
        files = files[:MAX_MASKS_FOR_FULL_SCAN]

    img_wh = []
    unique_value_counter = Counter()
    class_pixel_counter = Counter()
    class_image_counter = Counter()

    img_dir = ROOT / split / "img"
    mask_dir = ROOT / split / "mask"

    for stem in files:
        img_path = None
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
            p = img_dir / f"{stem}{ext}"
            if p.exists():
                img_path = p
                break
        if img_path is None:
            continue

        with Image.open(img_path) as im:
            w, h = im.size
        img_wh.append((w, h))

        if mask_dir.exists():
            mask_path = mask_dir / f"{stem}.png"
            if mask_path.exists():
                mask = np.array(Image.open(mask_path))
                vals, counts = np.unique(mask, return_counts=True)
                present_non_bg = set()

                for v, c in zip(vals.tolist(), counts.tolist()):
                    unique_value_counter[v] += 1
                    class_pixel_counter[v] += c
                    if v != background_id:
                        present_non_bg.add(v)

                for v in present_non_bg:
                    class_image_counter[v] += 1

                # lưu vài sample để visualize theo lớp hiếm/nhiều sau
                if len(sample_records[split]) < 20:
                    sample_records[split].append((stem, img_path, mask_path, mask))

    widths = [x[0] for x in img_wh]
    heights = [x[1] for x in img_wh]

    img_size_stats[split] = {
        "n": len(img_wh),
        "w_min": min(widths) if widths else None,
        "w_max": max(widths) if widths else None,
        "w_mean": float(np.mean(widths)) if widths else None,
        "h_min": min(heights) if heights else None,
        "h_max": max(heights) if heights else None,
        "h_mean": float(np.mean(heights)) if heights else None,
    }
    mask_value_stats[split] = unique_value_counter
    pixel_counts_per_class[split] = class_pixel_counter
    image_counts_per_class[split] = class_image_counter

    print(f"[{split}]")
    print("  image size stats:", img_size_stats[split])
    if len(unique_value_counter) > 0:
        print("  unique mask ids present:", sorted(unique_value_counter.keys())[:20], "..." if len(unique_value_counter)>20 else "")
        print("  num unique ids in masks:", len(unique_value_counter))
        print("  background pixel count :", class_pixel_counter.get(background_id, 0))
        non_bg_total = sum(c for k, c in class_pixel_counter.items() if k != background_id)
        bg = class_pixel_counter.get(background_id, 0)
        total = bg + non_bg_total
        print("  bg ratio              :", round(bg / max(total, 1), 4))
    else:
        print("  No mask stats (mask dir missing or empty)")
    print()

# =========================
# 5) TOP / RARE CLASSES
# =========================
print("=" * 100)
print("CLASS DISTRIBUTION (TRAIN)")
print("=" * 100)

train_pixel_counter = pixel_counts_per_class.get("train", Counter())
train_image_counter = image_counts_per_class.get("train", Counter())

rows = []
for cid in range(num_classes):
    cname = "background" if cid == background_id else id_to_class.get(cid, f"class_{cid}")
    rows.append({
        "id": cid,
        "class": cname,
        "pixels": int(train_pixel_counter.get(cid, 0)),
        "images": int(train_image_counter.get(cid, 0)) if cid != background_id else len(split_files.get("train", [])),
    })

rows_sorted_pixels = sorted(rows, key=lambda x: x["pixels"], reverse=True)
rows_sorted_images = sorted(rows, key=lambda x: x["images"], reverse=True)

print("Top 15 classes by pixel count:")
for r in rows_sorted_pixels[:15]:
    print(f"  id={r['id']:>3} | {r['class']:<24} | pixels={r['pixels']:<12} | images={r['images']}")

print("\nBottom 15 non-background classes by pixel count:")
non_bg_rows = [r for r in rows if r["id"] != background_id]
for r in sorted(non_bg_rows, key=lambda x: x["pixels"])[:15]:
    print(f"  id={r['id']:>3} | {r['class']:<24} | pixels={r['pixels']:<12} | images={r['images']}")

# =========================
# 6) VISUALIZE SAMPLES
# =========================
def colorize_mask(mask, num_classes, seed=0):
    rng = np.random.default_rng(seed)
    colors = rng.integers(0, 255, size=(num_classes, 3), dtype=np.uint8)
    colors[background_id] = np.array([0, 0, 0], dtype=np.uint8)
    return colors[mask]

def overlay_mask(image, mask_rgb, alpha=0.45):
    return (image * (1 - alpha) + mask_rgb * alpha).astype(np.uint8)

print("\n" + "=" * 100)
print("SAMPLE VISUALIZATION")
print("=" * 100)

for split in SPLITS:
    samples = sample_records.get(split, [])
    if not samples:
        continue

    picks = samples[:min(N_VIS, len(samples))]
    fig, axes = plt.subplots(len(picks), 4, figsize=(16, 4 * len(picks)))
    if len(picks) == 1:
        axes = np.expand_dims(axes, 0)

    for r, (stem, img_path, mask_path, mask) in enumerate(picks):
        img = np.array(Image.open(img_path).convert("RGB"))
        mask_rgb = colorize_mask(mask, num_classes=num_classes, seed=SEED)
        over = overlay_mask(img, mask_rgb)

        present_ids = sorted(np.unique(mask).tolist())
        present_names = [("background" if x == background_id else id_to_class.get(x, str(x))) for x in present_ids]

        axes[r, 0].imshow(img)
        axes[r, 0].set_title(f"{split} image\n{stem}")
        axes[r, 1].imshow(mask, cmap="nipy_spectral", vmin=0, vmax=num_classes-1)
        axes[r, 1].set_title("mask ids")
        axes[r, 2].imshow(mask_rgb)
        axes[r, 2].set_title("colorized mask")
        axes[r, 3].imshow(over)
        axes[r, 3].set_title("overlay")

        for c in range(4):
            axes[r, c].axis("off")

        print(f"[{split}] {stem}")
        print("  image:", img.shape, "mask:", mask.shape, "unique ids:", present_ids[:20], "..." if len(present_ids)>20 else "")
        print("  classes:", present_names[:15], "..." if len(present_names)>15 else "")

    plt.tight_layout()
    plt.show()

# =========================
# 7) PYTORCH DATASET
# =========================
class FoodSeg103RasterDataset(Dataset):
    def __init__(self, root, split="train", file_stems=None, image_transform=None, mask_transform=None):
        self.root = Path(root)
        self.split = split
        self.img_dir = self.root / split / "img"
        self.mask_dir = self.root / split / "mask"
        self.file_stems = sorted(file_stems) if file_stems is not None else self._discover()
        self.image_transform = image_transform
        self.mask_transform = mask_transform

    def _discover(self):
        img_stems = {p.stem for p in self.img_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]}
        mask_stems = {p.stem for p in self.mask_dir.glob("*.png")}
        return sorted(img_stems & mask_stems)

    def __len__(self):
        return len(self.file_stems)

    def _find_image_path(self, stem):
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
            p = self.img_dir / f"{stem}{ext}"
            if p.exists():
                return p
        raise FileNotFoundError(f"Không tìm thấy image cho stem={stem}")

    def __getitem__(self, idx):
        stem = self.file_stems[idx]
        img_path = self._find_image_path(stem)
        mask_path = self.mask_dir / f"{stem}.png"

        image = np.array(Image.open(img_path).convert("RGB"))
        mask = np.array(Image.open(mask_path), dtype=np.uint8)

        # optional external transforms
        if self.image_transform is not None:
            image = self.image_transform(image)
        if self.mask_transform is not None:
            mask = self.mask_transform(mask)

        # default -> tensor
        if not torch.is_tensor(image):
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        if not torch.is_tensor(mask):
            mask = torch.from_numpy(mask.astype(np.int64))

        return {
            "image": image,
            "mask": mask,
            "stem": stem,
        }

# train/val split from train only
train_stems_all = split_files.get("train", [])
if MAKE_VAL_FROM_TRAIN and len(train_stems_all) > 0:
    rng = random.Random(SEED)
    train_stems_all = train_stems_all.copy()
    rng.shuffle(train_stems_all)
    n_val = max(1, int(len(train_stems_all) * VAL_RATIO))
    val_stems = sorted(train_stems_all[:n_val])
    train_stems = sorted(train_stems_all[n_val:])
else:
    train_stems = sorted(train_stems_all)
    val_stems = []

train_ds = FoodSeg103RasterDataset(ROOT, split="train", file_stems=train_stems)
val_ds = FoodSeg103RasterDataset(ROOT, split="train", file_stems=val_stems) if val_stems else None
test_ds = FoodSeg103RasterDataset(ROOT, split="test", file_stems=split_files.get("test", [])) if (ROOT / "test" / "mask").exists() else None

print("\n" + "=" * 100)
print("PYTORCH DATASET CHECK")
print("=" * 100)
print("len(train_ds):", len(train_ds))
print("len(val_ds)  :", len(val_ds) if val_ds is not None else 0)
print("len(test_ds) :", len(test_ds) if test_ds is not None else "test mask missing / not built")

if len(train_ds) > 0:
    sample = train_ds[0]
    print("sample keys :", sample.keys())
    print("image shape :", tuple(sample["image"].shape), sample["image"].dtype, f"min={sample['image'].min().item():.4f}", f"max={sample['image'].max().item():.4f}")
    print("mask shape  :", tuple(sample["mask"].shape), sample["mask"].dtype)
    print("mask unique :", torch.unique(sample["mask"])[:20])

# =========================
# 8) RECOMMEND INITIAL TRAIN CONFIG FROM OBSERVED DATA
# =========================
print("\n" + "=" * 100)
print("RECOMMENDED INITIAL TRAIN CONFIG (BASED ON CURRENT DATA COPY)")
print("=" * 100)

train_mean_w = img_size_stats["train"]["w_mean"] or 512
train_mean_h = img_size_stats["train"]["h_mean"] or 512
short_side_mean = min(train_mean_w, train_mean_h)

if short_side_mean >= 700:
    suggested_crop = 640
elif short_side_mean >= 500:
    suggested_crop = 512
else:
    suggested_crop = 384

bg_pixels = train_pixel_counter.get(background_id, 0)
fg_pixels = sum(v for k, v in train_pixel_counter.items() if k != background_id)
bg_ratio = bg_pixels / max(bg_pixels + fg_pixels, 1)

rare_classes = [r for r in non_bg_rows if r["images"] <= max(3, int(0.01 * max(len(train_stems), 1)))]
many_rare = len(rare_classes) > 10

print(f"- Current copy is a subset-sized setup: train={len(train_stems)} (+val={len(val_stems)}) and test={len(split_files.get('test', []))}.")
print(f"- num_classes = {num_classes} (103 ingredients + background={background_id})")
print(f"- mean train image size ≈ ({train_mean_w:.1f}, {train_mean_h:.1f})")
print(f"- suggested crop size = {suggested_crop}")
print(f"- background pixel ratio ≈ {bg_ratio:.3f}")
print(f"- rare classes (very few training images) = {len(rare_classes)}")

print("\nSuggested baseline config:")
print({
    "num_classes": num_classes,
    "background_id": background_id,
    "ignore_index": None,   # dùng None trước; chỉ dùng 255 nếu bạn có vùng ignore riêng
    "model_baseline": "DeepLabV3-ResNet50 or BiSeNetV1-ResNet18",
    "input_crop": suggested_crop,
    "train_aug": {
        "random_scale": [0.5, 2.0],
        "random_crop": suggested_crop,
        "horizontal_flip": 0.5,
        "color_jitter": True,
        "normalize": "ImageNet mean/std",
        "mask_resize": "nearest"
    },
    "optimizer": "SGD",
    "lr": 0.01 if suggested_crop <= 512 else 0.005,
    "momentum": 0.9,
    "weight_decay": 5e-4,
    "scheduler": "poly",
    "poly_power": 0.9,
    "batch_size_hint": "8-16 @512 crop, 4-8 @640 crop (tùy VRAM)",
    "loss": "CrossEntropy",
    "extra_if_long_tail_is_bad": ["class weights", "Focal Loss", "Lovasz-Softmax"],
    "metrics": ["mIoU", "mAcc", "aAcc"],
})

if many_rare:
    print("\nObservation:")
    print("- Long-tail is likely noticeable in this subset copy.")
    print("- Start with plain CE, but be ready to add class weights or Focal/Lovasz if rare-class IoU collapses.")
if bg_ratio > 0.75:
    print("- Background dominates many pixels; watch for degenerate predictions toward background.")
print("- Use validation split from train for model selection; do not tune on test.")

DATASET ROOT: F:\ANHTHU\1-HCMUS\1 - STUDY\HKVIII\CV\PROJECT\source\dataset\foodseg103
[train]
  img : dataset\foodseg103\train\img exists= True
  ann : dataset\foodseg103\train\ann exists= True
  mask: dataset\foodseg103\train\mask exists= True
  #img : 922
  #ann : 922
  #mask: 922

[test]
  img : dataset\foodseg103\test\img exists= True
  ann : dataset\foodseg103\test\ann exists= True
  mask: dataset\foodseg103\test\mask exists= True
  #img : 399
  #ann : 399
  #mask: 399

META / CLASS MAPPING
meta.json found
  num meta classes : 103
  num meta tags    : 15
  first 10 meta class titles: ['almond', 'apple', 'apricot', 'asparagus', 'avocado', 'bamboo shoots', 'banana', 'bean sprouts', 'biscuit', 'blueberry']
class_mapping.json found
  num ingredient classes : 103
  background_id          : 103
  num_classes(total)     : 104
  first 10 classes: [('almond', 0), ('apple', 1), ('apricot', 2), ('asparagus', 3), ('avocado', 4), ('bamboo shoots', 5), ('banana', 6), ('bean sprouts', 7), ('bisc

KeyboardInterrupt: 

In [ ]:
# One-cell dataset audit + sample visualization + class analysis + PyTorch Dataset
# Works for: source/dataset/foodseg103/{train,test}/{img,ann,mask}

# TƯƠNG THÍCH MÔI TRƯỜNG COLAB (1x GPU T4, 12GB RAM)
import os, gc, random
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF

ROOT = Path("/dataset/foodseg103")
if not ROOT.exists():
    ROOT = Path("dataset/foodseg103")

# ==============================================================================
# MODULE 1: BẢNG ĐIỀU KHIỂN TRUNG TÂM (CENTRAL CONFIGURATION)
# ==============================================================================
CONFIG = {
    # Thay đổi đường dẫn vật lý phù hợp với vị trí bạn lưu data trên Colab
    "data_root": ROOT,
    "save_dir": "/content/checkpoints",
    "seed": 42,

    "num_classes": 104,
    "img_size": (512, 512),
    "compute_class_weights": True,

    "batch_size": 8,
    "epochs": 80,
    "lr": 0.01,
    "momentum": 0.9,
    "weight_decay": 5e-4,
    "poly_power": 0.9
}

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.backends.cudnn.benchmark = True

# ==============================================================================
# MODULE 2: HỒ SƠ HÓA VÀ TIỀN XỬ LÝ (DATA PIPELINE)
# ==============================================================================
def compute_class_weights(mask_dir, num_classes, c=1.02):
    print("[Data Profiler] Đang quét phân phối điểm ảnh (ENet Method)...")
    pixel_counts = np.zeros(num_classes, dtype=np.int64)
    mask_files = [f for f in os.listdir(mask_dir) if f.endswith('.png')]

    for mask_file in tqdm(mask_files, desc="Đếm Pixel"):
        mask_path = os.path.join(mask_dir, mask_file)
        mask_np = np.array(Image.open(mask_path))

        valid_pixels = mask_np[(mask_np >= 0) & (mask_np < num_classes)]
        counts = np.bincount(valid_pixels, minlength=num_classes)
        pixel_counts += counts

    total_pixels = pixel_counts.sum()
    freq = pixel_counts / (total_pixels + 1e-6)

    weights = 1.0 / np.log(c + freq)
    weights[pixel_counts == 0] = 0.0
    weights = weights / weights.sum() * num_classes
    print("[Data Profiler] Hoàn tất toán học trọng số.")

    return torch.from_numpy(weights).float()


class FoodSegDataset(Dataset):
    def __init__(self, root_dir, split="train", target_size=(512, 512), augment=False):
        self.split_dir = Path(root_dir) / split
        self.img_dir = self.split_dir / "img"
        self.mask_dir = self.split_dir / "mask"
        self.target_size = target_size
        self.augment = augment

        valid_exts = {'.jpg', '.png', '.jpeg'}
        self.images = sorted([f for f in os.listdir(self.img_dir) if os.path.splitext(f)[1].lower() in valid_exts])

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        base_name = os.path.splitext(self.images[idx])[0]
        image = Image.open(self.img_dir / self.images[idx]).convert("RGB")
        mask = Image.open(self.mask_dir / f"{base_name}.png").convert("L")

        image = TF.resize(image, self.target_size, interpolation=TF.InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.target_size, interpolation=TF.InterpolationMode.NEAREST)

        if self.augment and random.random() > 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)

        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))

        return {"image": image, "mask": mask}


# ==============================================================================
# MODULE 3: KIẾN TRÚC MẠNG NƠ-RON VÀ HÀM MẤT MÁT (MODEL & LOSS)
# ==============================================================================
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, ignore_index=255):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=weight, ignore_index=ignore_index, reduction='none')
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = self.ce(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


# Toàn vẹn Đồ thị tính toán BiSeNetV2
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, groups=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.block(x)


class DWConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = ConvBNReLU(in_ch, in_ch, k=3, s=stride, p=1, groups=in_ch)
        self.pw = ConvBNReLU(in_ch, out_ch, k=1, s=1, p=0)
    def forward(self, x): return self.pw(self.dw(x))


class DetailBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.s1 = nn.Sequential(ConvBNReLU(3, 64, 3, 2, 1), ConvBNReLU(64, 64, 3, 1, 1))
        self.s2 = nn.Sequential(ConvBNReLU(64, 64, 3, 2, 1), ConvBNReLU(64, 64, 3, 1, 1), ConvBNReLU(64, 64, 3, 1, 1))
        self.s3 = nn.Sequential(ConvBNReLU(64, 128, 3, 2, 1), ConvBNReLU(128, 128, 3, 1, 1), ConvBNReLU(128, 128, 3, 1, 1))
    def forward(self, x): return self.s3(self.s2(self.s1(x)))


class SemanticBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            ConvBNReLU(3, 16, 3, 2, 1), DWConvBNReLU(16, 32, stride=2),
            DWConvBNReLU(32, 64, stride=2), DWConvBNReLU(64, 128, stride=2),
            DWConvBNReLU(128, 128, stride=1)
        )
        self.context = nn.Sequential(nn.AdaptiveAvgPool2d(1), ConvBNReLU(128, 128, 1, 1, 0))
    def forward(self, x):
        feat = self.stem(x)
        ctx = F.interpolate(self.context(feat), size=feat.shape[-2:], mode="bilinear", align_corners=False)
        return feat + ctx


class BGALayer(nn.Module):
    def __init__(self, detail_ch=128, semantic_ch=128, out_ch=128):
        super().__init__()
        self.detail_proj = ConvBNReLU(detail_ch, out_ch, 3, 1, 1)
        self.semantic_proj = ConvBNReLU(semantic_ch, out_ch, 3, 1, 1)
        self.fuse = ConvBNReLU(out_ch, out_ch, 3, 1, 1)
    def forward(self, detail, semantic):
        semantic = F.interpolate(semantic, size=detail.shape[-2:], mode="bilinear", align_corners=False)
        return self.fuse(self.detail_proj(detail) + self.semantic_proj(semantic))


class SegHead(nn.Module):
    def __init__(self, in_ch, mid_ch, num_classes, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            ConvBNReLU(in_ch, mid_ch, 3, 1, 1), nn.Dropout(dropout),
            nn.Conv2d(mid_ch, num_classes, kernel_size=1, bias=True)
        )
    def forward(self, x, out_size):
        return F.interpolate(self.block(x), size=out_size, mode="bilinear", align_corners=False)


class BiSeNetV2Tiny(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.detail = DetailBranch()
        self.semantic = SemanticBranch()
        self.bga = BGALayer(128, 128, 128)
        self.head = SegHead(128, 128, num_classes)
        self.aux = SegHead(128, 64, num_classes)
    def forward(self, x):
        out_size = x.shape[-2:]
        d, s = self.detail(x), self.semantic(x)
        main_logits = self.head(self.bga(d, s), out_size)
        if self.training:
            aux_logits = self.aux(s, out_size)
            return main_logits, aux_logits
        return main_logits


# ==============================================================================
# ĐỊNH NGHĨA CÁC TOÁN TỬ ĐO LƯỜNG VÀ LUỒNG ĐIỀU KHIỂN BỘ NHỚ
# ==============================================================================
@torch.no_grad()
def update_hist(hist, pred, target, num_classes):
    mask = (target >= 0) & (target < num_classes)
    x = num_classes * target[mask] + pred[mask]
    binc = torch.bincount(x.reshape(-1), minlength=num_classes**2)
    hist += binc.reshape(num_classes, num_classes)
    return hist


@torch.no_grad()
def compute_metrics_from_hist(hist):
    hist = hist.float()
    eps = 1e-6
    tp = torch.diag(hist)
    gt = hist.sum(dim=1)
    pred = hist.sum(dim=0)

    iou = tp / (gt + pred - tp + eps)
    acc_cls = tp / (gt + eps)
    aacc = tp.sum() / (hist.sum() + eps)

    return {
        "mIoU": torch.nanmean(iou).item(),
        "mAcc": torch.nanmean(acc_cls).item(),
        "aAcc": aacc.item()
    }


def train_one_epoch(model, dataloader, criterion, optimizer, scheduler, scaler, device, epoch):
    model.train()
    train_loss_sum = 0.0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Train]", leave=False)
    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True).long()

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda', enabled=True):
            main_logits, aux_logits = model(images)
            loss_main = criterion(main_logits, masks)
            loss_aux = criterion(aux_logits, masks)
            loss = loss_main + 0.4 * loss_aux

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_loss_sum += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.6f}")

    return train_loss_sum / len(dataloader)


@torch.no_grad()
def validate_one_epoch(model, dataloader, criterion, device, num_classes, epoch):
    model.eval()
    val_loss_sum = 0.0
    hist = torch.zeros((num_classes, num_classes), device=device, dtype=torch.int64)

    pbar = tqdm(dataloader, desc=f"Epoch {epoch} [Val]", leave=False)
    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True).long()

        with torch.amp.autocast('cuda', enabled=True):
            main_logits = model(images)
            loss = criterion(main_logits, masks)

        val_loss_sum += loss.item()
        preds = torch.argmax(main_logits, dim=1)
        hist = update_hist(hist, preds, masks, num_classes)

    metrics = compute_metrics_from_hist(hist)
    return val_loss_sum / len(dataloader), metrics


# ==============================================================================
# MODULE 4: LUỒNG THỰC THI CHÍNH (MAIN EXECUTION PIPELINE)
# ==============================================================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[*] Kiến trúc phần cứng: {device} | Backend: Colab T4")

    Path(CONFIG["save_dir"]).mkdir(parents=True, exist_ok=True)

    train_dataset = FoodSegDataset(CONFIG["data_root"], split="train", target_size=CONFIG["img_size"], augment=True)
    val_dataset = FoodSegDataset(CONFIG["data_root"], split="test", target_size=CONFIG["img_size"], augment=False)

    # Windows + notebook: dùng single-process DataLoader để tránh worker crash.
    loader_kwargs = {
        "batch_size": CONFIG["batch_size"],
        "num_workers": 0,
        "pin_memory": device.type == "cuda",
    }

    train_loader = DataLoader(train_dataset, shuffle=True, drop_last=True, **loader_kwargs)
    val_loader = DataLoader(val_dataset, shuffle=False, drop_last=False, **loader_kwargs)

    class_weights = None
    if CONFIG["compute_class_weights"]:
        mask_dir = Path(CONFIG["data_root"]) / "train" / "mask"
        class_weights = compute_class_weights(mask_dir, CONFIG["num_classes"]).to(device)
        print(f"[*] Trọng số điều chuẩn class 103 (Background): {class_weights[103]:.4f}")

    model = BiSeNetV2Tiny(num_classes=CONFIG["num_classes"]).to(device)

    # Cấu hình đa GPU dự phòng nếu đổi môi trường (Colab T4 chỉ có 1 GPU)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    criterion = FocalLoss(weight=class_weights, gamma=2.0, ignore_index=255).to(device)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=CONFIG["lr"],
        momentum=CONFIG["momentum"],
        weight_decay=CONFIG["weight_decay"]
    )

    scaler = torch.amp.GradScaler('cuda', enabled=True)

    total_iters = CONFIG["epochs"] * len(train_loader)
    poly_lambda = lambda it: (1 - it / total_iters) ** CONFIG["poly_power"]
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=poly_lambda)

    best_miou = -1.0
    print("[*] Sẵn sàng luồng dữ liệu. Bắt đầu lan truyền...")

    for epoch in range(1, CONFIG["epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, device, epoch)
        val_loss, metrics = validate_one_epoch(model, val_loader, criterion, device, CONFIG["num_classes"], epoch)

        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"mIoU: {metrics['mIoU']:.4f} | aAcc: {metrics['aAcc']:.4f}")

        # Xử lý trích xuất state_dict an toàn cho DataParallel
        model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        checkpoint = {
            "epoch": epoch,
            "model_state": model_state,
            "optimizer_state": optimizer.state_dict(),
            "best_miou": best_miou
        }

        torch.save(checkpoint, Path(CONFIG["save_dir"]) / "bisenetv2_last.pth")

        if metrics["mIoU"] > best_miou:
            best_miou = metrics["mIoU"]
            torch.save(checkpoint, Path(CONFIG["save_dir"]) / "bisenetv2_best.pth")
            print(f" -> Cập nhật cực trị mới: mIoU = {best_miou:.4f}")

        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()


main()